# D1.3 · Bonus — finding the agents, and keeping what they emit

**Function D — The Agentic SOC → Discover — the Sensors, and the Agent-Shaped Hole in Them**

Builds on **[D1.2 · Drift monitoring — behaviour that changes without a code change](https://spbreed.github.io/cyber-commons/lessons/D1.2.html)**.

| | |
|---|---|
| Tools used | OpenTelemetry, OpenSearch |

## What this lesson is

**What it covers.** Two halves of one job: scoring actors on timing, sequencing and volume to find the agents that are in no registry, then deciding per-field retention on the traces you inherit once you keep them.

**Why a security engineer needs it.** Shadow autonomy is invisible to a stack that reads an agent as the person whose credential it holds, and the threshold that finds it is set by cost rather than accuracy — a flagged human costs half an analyst-hour, a missed agent costs forty. The trace you then hold is the most useful log source you have and the most sensitive, because it carries the reasoning and whatever was in the context window.

This is a **control** lesson: it builds the mechanism, then breaks it, so you can see what the control is actually load-bearing for rather than taking the claim on trust.

## 1 · The hook

The agent holds a person's authority and acts in their name, so conventional UEBA reads it as that person behaving strangely at 03:00. Find it on behaviour instead, and you inherit its trace — which contains the reasoning, and whatever was in the context window when it ran.

> **At CyberTravels.** CyberTravels acts under Alex's authority and in Alex's name, so conventional UEBA reads it as Alex behaving strangely at 3am. Score it on behaviour instead and it is unmistakable — and then you are holding its trace, which carries prompts, tool calls, decisions and identities that appear in no application log CyberTravels has. R10, R11.

## 2 · The framework

```
   1 · WHO IS ACTING?              2 · WHAT YOU THEN HOLD

   the log says                    +----------------------------+
   +------------------+            | prompts     tool calls     |
   | user: dana@corp  |            | decisions   identities     |
   | action: deploy   |            +----------------------------+
   +------------------+                       |
           |                        none of it is in an
   UEBA: "dana, strangely"          application log, and the
           |                        prompts carry whatever was
   the missing field is not         in the context window
   "suspicious" - it is
   "actor_type", recoverable                  v
   from behaviour alone:            retention is decided per
   regularity, rate, continuity     FIELD, not per record

   find the actor first. its trace is what you then have to keep.
```

The two questions in this lesson are one question asked twice: **which of the
actors in your logs is software, and what does its trace contain once you keep
it?** Neither has an answer in a standard SIEM, and the second only becomes
urgent once the first one works.

### Finding the actor

The agents you most need to find are the ones in no registry (A3.7), and they
act under a person's authority in a person's name — so conventional UEBA reads
them as that person behaving strangely. Three behavioural signals separate them,
none sufficient alone:

- **Regularity** — the coefficient of variation of inter-arrival times. People
  are irregular; loops are metronomic.
- **Rate** — sustained multi-action-per-second activity is not typing.
- **Continuity** — software has no evenings.

The error directions are not symmetric, and that is what sets the threshold. A
**human misclassified as an agent** triggers an investigation: mild, and
self-correcting. An **agent misclassified as human** stays invisible, which is
the entire risk you were trying to address. Cost-weighting therefore picks a
lower threshold than accuracy-maximisation would.

### Keeping its trace

Once you have found it, agent telemetry turns out to have a property no other
log source has: it contains the **reasoning**, not just the action. The trace
records what the agent was trying to do, what it considered, and what the
verifier said.

That is enormously useful for investigation and it is a retention and privacy
problem, because reasoning traces contain whatever was in the context window —
routinely customer data, source code and secrets the agent read legitimately.
So retention is decided **per field**, not per record:

| Field | Forensic value | Sensitivity |
|---|---|---|
| timestamps, tool, target | high | low |
| verifier detail | high | low |
| acting identity + chain | high | low |
| model prompts | medium | **high** |
| tool results | high | **high** |

The first three are cheap and should be kept long. The last two are where the
retention conversation actually is.

> **Anchor → D1.0.** Both halves set the floor for all five intervals. An agent nobody registered has no clock running but its own behaviour, and on data that was never emitted discover does not take a long time — it never completes. The retention decision is the same argument backwards: it fixes how far into the past an investigation is allowed to reach.

## 3 · Finding the actor, as a skill

The skill scores five actors on behaviour rather than on what they claim to be, sweeps the threshold, and then picks it by expected cost — because a flagged human costs half an analyst-hour and a missed agent costs forty.

### The skill — [`skills/detection/agent-versus-human-scoring/SKILL.md`](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/skills/detection/agent-versus-human-scoring/SKILL.md)

```yaml
name: agent-versus-human-scoring
description: >-
  Score actors on behavioural signals to separate agents from people, sweep the
  threshold, and pick it by expected cost rather than by accuracy. Use when
  deciding whether a session is automated, or when unregistered automation needs
  finding.
allowed-tools: Read, Grep, Glob
```

# Pick the threshold by what each mistake costs

Separating agent from human is a scoring problem with two asymmetric errors: a
flagged human costs an analyst half an hour, and a missed agent costs whatever
an unmonitored automation does. Choosing the threshold by accuracy weights those
equally, which is the one thing you know is wrong.

## When to use this

Finding unregistered automation, deciding whether a session is a person, and
before any control that treats agents differently from users.

## Procedure

**1 — Score on behaviour, not on the user agent string.** Inter-action variance,
rate, breadth, and the share of actions with no preceding read. Anything
self-declared is a claim.

**2 — Score a spread of real actors.** A service indexer, an unknown token, a
person, a person driving an IDE assistant, and an agent deliberately jittered to
look human. The last two are the interesting middle.

**3 — Sweep the threshold and record both errors.** Humans flagged and agents
missed, at each setting. They move in opposite directions and the crossing point
is not the answer.

**4 — Attach a cost to each error and minimise the total.** Analyst hours for a
false positive, expected hours of an unmonitored agent for a false negative. The
chosen threshold now has a justification somebody can argue with.

**5 — Join to the registry.** An actor scoring as an agent and absent from the
registry is the finding worth routing; a registered agent scoring as an agent is
working correctly.

## Example

**Input** — the fixture committed at the top of [`scripts/agent_versus_human_scoring.py`](scripts/agent_versus_human_scoring.py). Edit it and re-run: the buckets, counts and verdicts below are derived from it, not hard-coded.

**Output** — the opening lines of a real run:

```
actor                   score     cv   rate/s   span_h  truth
--------------------------------------------------------------
svc-indexer             0.800    0.0    20.04     0.01  agent
dana@corp               0.028   1.55      0.0     1.11  human
unknown-token-7f3c      0.563    0.0      1.0     0.11  agent
sam@corp-ide            0.533    0.0      0.5      0.1  human
polite-agent            0.021   1.26      0.0     0.83  agent
 threshold  humans flagged   agents MISSED
```

The run continues past this. The script is the example: `test_skills.py` executes it on every build, so this block cannot drift from what the skill actually prints.

## Output contract

```json
{
  "actors": [{"name": "str", "score": 0.0, "truth": "agent|human|unknown"}],
  "sweep": [{"threshold": 0.0, "humans_flagged": 0, "agents_missed": 0, "expected_cost": 0.0}],
  "costs": {"false_positive_hours": 0.0, "false_negative_hours": 0.0},
  "chosen": {"threshold": 0.0, "why": "str"},
  "registry": {"scored_agent_unregistered": ["str"]}
}
```

## Failure modes

- **Scoring the user agent string.** It is self-declared.
- **Optimising accuracy.** It assumes the two errors cost the same.
- **Flagging registered agents.** They are supposed to look like agents.

In [ ]:
# The code is not in this notebook. It is this file in the repository:
#   https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/skills/detection/agent-versus-human-scoring/scripts/agent_versus_human_scoring.py
SCRIPT = "skills/detection/agent-versus-human-scoring/scripts/agent_versus_human_scoring.py"
REPO = "https://github.com/spbreed/cyber-commons"
BRANCH = "claude/vulnbench-setup-scheduling-81aqov"

import glob, os, subprocess, sys

CLONE = "/kaggle/working/cyber-commons"
_root = next((r for r in (".", "..", "../..", CLONE)
              if os.path.isfile(os.path.join(r, SCRIPT))), None)

if _root is None:
    # --filter=blob:none --sparse fetches the tree without the history or the
    # notebooks; sparse-checkout then materialises only the two directories a
    # lesson needs: the procedures, and the repository they are run against.
    _c = subprocess.run(["git", "clone", "--depth", "1", "--filter=blob:none",
                         "--sparse", "--branch", BRANCH, REPO, CLONE],
                        capture_output=True, text=True)
    if _c.returncode:
        raise SystemExit(
            "could not fetch the skills: " + _c.stderr.strip()[-300:] +
            "\nOn Kaggle this needs Internet on in the notebook settings, which "
            "needs a phone-verified account. Without one, attach the dataset "
            "cybercommons/cyber-commons-skills instead — it holds the same tree.")
    # `skills` is the procedures; `cybertravels` is the sample repository they
    # scan; `curriculum` and `site/data` hold the framework mapping and the
    # session list that the reference-lookup skill reads. Miss any of them and
    # the skill clones successfully and then fails on a path that is not there,
    # which is how A0.2 failed its first Kaggle run.
    subprocess.run(["git", "-C", CLONE, "sparse-checkout", "set",
                    "skills", "cybertravels", "curriculum", "site/data"],
                   capture_output=True, text=True)
    _root = CLONE

_out = subprocess.run([sys.executable, os.path.join(_root, SCRIPT)],
                      capture_output=True, text=True,
                      env=dict(os.environ,
                               PYTHONPATH=os.path.join(_root, "skills/_runtime"),
                               PYTHONHASHSEED="0"))
print(_out.stdout, end="")
if _out.returncode:
    raise SystemExit(_out.stderr.strip()[-2000:])

## 4 · Keeping the trace, as a skill

The run record contains a payment-card pattern, in a source file the agent read legitimately. The skill scans every field, then sets retention per field so timestamps and verdicts survive for 400 days and prompts do not survive 30.

### The skill — [`skills/detection/agent-telemetry-retention/SKILL.md`](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/skills/detection/agent-telemetry-retention/SKILL.md)

```yaml
name: agent-telemetry-retention
description: >-
  Scan an agent's run record for sensitive content it read legitimately, then
  set retention per field rather than per record so the investigable parts
  survive and the prompts do not. Use when agent telemetry is being kept,
  discarded, or argued about with privacy.
allowed-tools: Read, Grep, Glob
```

# The agent read a card number because you asked it to

An agent run record is the richest telemetry in the estate and the most
dangerous to keep whole. The content it read — source files, tickets, documents
— lands in the record, so the record inherits every classification the source
had. Per-record retention forces a choice between losing the investigation and
keeping the data; per-field does not.

## When to use this

Before agent telemetry is retained at scale, and whenever a retention policy is
being written by someone who has not read a run record.

## Procedure

**1 — Read one full record.** Every step, every field. This is the step people
skip, and it is where the payment-card pattern in a legitimately-read source
file turns up.

**2 — Scan for sensitive patterns across all fields.** Card numbers, keys,
personal data, health terms. Record which field carried each hit — prompts and
tool results are the usual answer.

**3 — Classify fields by investigative value.** Timestamps, tool name, target
and verifier verdict answer most investigation questions. Prompts and raw tool
output answer few and carry most of the risk.

**4 — Set retention per field.** Long for the structural fields, short for the
content ones. Then age a record and check what an investigation could still do
with it — that check is what makes the policy defensible.

**5 — State what is lost.** A short prompt retention means you cannot re-derive
motivation after that window. Say so, rather than discovering it in an incident.

## Example

**Input** — the fixture committed at the top of [`scripts/agent_telemetry_retention.py`](scripts/agent_telemetry_retention.py). Edit it and re-run: the buckets, counts and verdicts below are derived from it, not hard-coded.

**Output** — the opening lines of a real run:

```
full record (everything the harness saw):
    {'n': 1, 'tool': 'read_file', 'target': '/work/repo/billing.py', 'verifier': 'n/a', 'ok': True, 'prompt': 'Investigate finding SEC-4471 in billing.py', 'result': 'def charge(card_number, amount):  # card_number = 4111111111111111'}
    {'n': 2, 'tool': 'search_code', 'target': 'charge(', 'verifier': 'n/a', 'ok': True, 'prompt': 'find callers of charge()', 'result': 'api/checkout.py:88 charge(user.card, total)'}
    {'n': 3, 'tool': 'write_file', 'target': '/work/repo/billing.py', 'verifier': 'tests pass', 'ok': True, 'prompt': 'apply the fix', 'result': 'patch applied'}
sensitive content found in the trace:
   step 1  result   payment card

Nobody put a card number in the trace deliberately. The agent read a
```

The run continues past this. The script is the example: `test_skills.py` executes it on every build, so this block cannot drift from what the skill actually prints.

## Output contract

```json
{
  "record": {"steps": 0, "fields": ["str"]},
  "scan": [{"pattern": "str", "field": "str", "legitimate_source": "str"}],
  "retention": [{"field": "str", "days": 0, "investigative_value": "high|low"}],
  "aged_record": {"age_days": 0, "questions_still_answerable": ["str"], "lost": ["str"]}
}
```

## Failure modes

- **Retaining or discarding whole records.** Both answers are wrong.
- **Scanning prompts only.** Tool results carry the same content.
- **Not stating what the short windows lose.** That is the trade being made.

In [ ]:
# The code is not in this notebook. It is this file in the repository:
#   https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/skills/detection/agent-telemetry-retention/scripts/agent_telemetry_retention.py
SCRIPT = "skills/detection/agent-telemetry-retention/scripts/agent_telemetry_retention.py"
REPO = "https://github.com/spbreed/cyber-commons"
BRANCH = "claude/vulnbench-setup-scheduling-81aqov"

import glob, os, subprocess, sys

CLONE = "/kaggle/working/cyber-commons"
_root = next((r for r in (".", "..", "../..", CLONE)
              if os.path.isfile(os.path.join(r, SCRIPT))), None)

if _root is None:
    # --filter=blob:none --sparse fetches the tree without the history or the
    # notebooks; sparse-checkout then materialises only the two directories a
    # lesson needs: the procedures, and the repository they are run against.
    _c = subprocess.run(["git", "clone", "--depth", "1", "--filter=blob:none",
                         "--sparse", "--branch", BRANCH, REPO, CLONE],
                        capture_output=True, text=True)
    if _c.returncode:
        raise SystemExit(
            "could not fetch the skills: " + _c.stderr.strip()[-300:] +
            "\nOn Kaggle this needs Internet on in the notebook settings, which "
            "needs a phone-verified account. Without one, attach the dataset "
            "cybercommons/cyber-commons-skills instead — it holds the same tree.")
    # `skills` is the procedures; `cybertravels` is the sample repository they
    # scan; `curriculum` and `site/data` hold the framework mapping and the
    # session list that the reference-lookup skill reads. Miss any of them and
    # the skill clones successfully and then fails on a path that is not there,
    # which is how A0.2 failed its first Kaggle run.
    subprocess.run(["git", "-C", CLONE, "sparse-checkout", "set",
                    "skills", "cybertravels", "curriculum", "site/data"],
                   capture_output=True, text=True)
    _root = CLONE

_out = subprocess.run([sys.executable, os.path.join(_root, SCRIPT)],
                      capture_output=True, text=True,
                      env=dict(os.environ,
                               PYTHONPATH=os.path.join(_root, "skills/_runtime"),
                               PYTHONHASHSEED="0"))
print(_out.stdout, end="")
if _out.returncode:
    raise SystemExit(_out.stderr.strip()[-2000:])

## What you just proved

First: the service indexer and unknown token score highest, the human lowest, with the IDE user and the politely-jittered agent in between. The threshold sweep shows humans flagged rising and agents missed falling as it drops, cost-weighting selects a low one, and joining against the registry names the unregistered actors as shadow agents. Then the trace of one of them: a payment-card pattern in a source file it read legitimately, and per-field retention that keeps timestamps, tool, target and verifier for 400 days while dropping prompts at 30 and tool results at 7. After 90 days no sensitive content remains and the record still answers what the agent did.

## Your turn

Run the scoring against a week of your own authentication logs and count the actors it flags that are not in your registry. Then check the retention period on whatever traces you keep for them: if it matches your firewall logs, one of those two numbers was chosen without anyone looking at what the traces contain.

## Where this leaves you

**What you can do now.** You can measure the discover interval instead of assuming it: four sensor classes scored against what an agent actually does, drift caught without a code change, and — as a bonus — the agents nobody registered found on behaviour, with their traces kept per field.

**What you still cannot do.** Four of the nine ordinary agent actions are seen by nothing you own, and the source that would see them lands nowhere. Everything here is a finding in a notebook; the detect interval is exactly where it was.

**Chapter D2 builds the place it lands and the rules that read it: the lake, tiered by the queries the SOC runs, and detections mapped to ATT&CK and ATLAS.**

---

**Next → [D2.1 · The detection data lake — where agent telemetry lands](https://spbreed.github.io/cyber-commons/lessons/D2.1.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/D1.3.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/D1.3.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*